# Multi-Strategy Toolpath: AI-Driven Layer-Wise Deposition

**PhD Thesis — Diana Paola Ayala Roldán**

## Core Idea

Different wound layers require different deposition strategies:
- **Deep layers** → Honeycomb (structural integrity, fast fill)
- **Surface layers** → Geodesic (conformal, matches tissue curvature)
- **Transition layers** → Hybrid (honeycomb core + geodesic perimeter)

This notebook demonstrates the **StrategyRouter** that classifies each layer and generates the appropriate toolpath.

**No GPU required** — this is a planning/visualization notebook.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.patches import RegularPolygon, FancyArrowPatch
from matplotlib.collections import PatchCollection
from pathlib import Path
from mpl_toolkits.mplot3d import Axes3D

# Add repo src to path
sys.path.insert(0, str(Path('..') / 'src'))

from multistrategy.toolpath.strategy_router import StrategyRouter, StrategyConfig, LayerStrategy
from multistrategy.toolpath.honeycomb_planner import HoneycombPlanner
from multistrategy.toolpath.geodesic_planner import GeodesicPlanner
from multistrategy.toolpath.hybrid_planner import HybridPlanner

Path('figures').mkdir(exist_ok=True)
print('Multi-Strategy Toolpath loaded successfully')

## 1. Synthetic Wound Input

We simulate a realistic wound with varying depth — deep center, shallow edges — to demonstrate how the strategy router classifies each layer.

In [ ]:
# Generate synthetic wound boundary and layer amounts
np.random.seed(42)
NUM_ANGLES = 64
NUM_LAYERS = 6

# Elliptical wound boundary (mm)
angles = np.linspace(0, 2 * np.pi, NUM_ANGLES, endpoint=False)
a, b = 25.0, 18.0  # semi-axes
radii_mm = (a * b) / np.sqrt((b * np.cos(angles))**2 + (a * np.sin(angles))**2)
radii_mm += np.random.normal(0, 0.8, NUM_ANGLES)  # slight irregularity
boundary_mm = np.stack([radii_mm * np.cos(angles), radii_mm * np.sin(angles)], axis=1)

# Layer amounts: deep layers have high fill, surface layers have low fill
# This mimics what PolarDecoder3DLayered would output
layer_amounts = np.zeros((NUM_LAYERS, NUM_ANGLES))
for i in range(NUM_LAYERS):
    # Deeper layers (lower index) = higher fill fraction
    base_fill = 1.0 - (i / NUM_LAYERS) * 0.8
    # Spatial variation: center fills more than edges
    spatial_var = 0.8 + 0.2 * np.cos(angles * 2)
    layer_amounts[i] = base_fill * spatial_var + np.random.normal(0, 0.05, NUM_ANGLES)
    layer_amounts[i] = np.clip(layer_amounts[i], 0.05, 1.0)

print(f'Wound boundary: {NUM_ANGLES} points, ~{np.ptp(radii_mm):.1f}mm diameter variation')
print(f'Layer amounts shape: {layer_amounts.shape}')
print(f'Layer mean fills: {[f"{la.mean():.2f}" for la in layer_amounts]}')

## 2. Strategy Router: Layer Classification

The router examines each layer's fill amount and classifies it as HONEYCOMB, GEODESIC, or HYBRID based on learned thresholds.

In [ ]:
# Configure and run the strategy router
config = StrategyConfig(
    structural_threshold=0.6,   # Above this → HONEYCOMB
    conformal_threshold=0.3,    # Below this → GEODESIC
    # Between thresholds → HYBRID
)

router = StrategyRouter(config)
plan = router.classify_layers(layer_amounts, boundary_mm)

print('\n╔══════════════════════════════════════════════════╗')
print('║     MULTI-STRATEGY LAYER CLASSIFICATION          ║')
print('╠══════════════════════════════════════════════════╣')
for lp in plan.layer_plans:
    emoji = {'HONEYCOMB': '🍯', 'GEODESIC': '🌊', 'HYBRID': '🔀'}[lp.strategy.name]
    bar = '█' * int(lp.mean_fill * 20)
    print(f'║  Layer {lp.layer_idx}: {emoji} {lp.strategy.name:<10} │ fill={lp.mean_fill:.2f} {bar}')
print('╚══════════════════════════════════════════════════╝')
print(f'\nTotal layers: {plan.num_layers}')
print(f'Honeycomb: {sum(1 for lp in plan.layer_plans if lp.strategy == LayerStrategy.HONEYCOMB)}')
print(f'Geodesic:  {sum(1 for lp in plan.layer_plans if lp.strategy == LayerStrategy.GEODESIC)}')
print(f'Hybrid:    {sum(1 for lp in plan.layer_plans if lp.strategy == LayerStrategy.HYBRID)}')

## 3. Visualization: Strategy Classification Map

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3a. STRATEGY CLASSIFICATION HEATMAP
# ─────────────────────────────────────────────────────────────────────────────

strategy_colors = {
    LayerStrategy.HONEYCOMB: '#FF8C00',
    LayerStrategy.GEODESIC: '#0077BE',
    LayerStrategy.HYBRID: '#8B008B',
}
strategy_labels = {
    LayerStrategy.HONEYCOMB: 'Honeycomb (structural)',
    LayerStrategy.GEODESIC: 'Geodesic (conformal)',
    LayerStrategy.HYBRID: 'Hybrid (transition)',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: Layer amounts heatmap with strategy overlay
ax = axes[0]
im = ax.imshow(layer_amounts, aspect='auto', cmap='YlOrRd', interpolation='nearest')
ax.set_xlabel('Point index (around boundary)')
ax.set_ylabel('Layer (0=deepest)')
ax.set_title('Layer Fill Amounts\n(from Volumetric Decoder)', fontsize=11, fontweight='bold')
fig.colorbar(im, ax=ax, label='Fill fraction')

# Add strategy labels on y-axis
for lp in plan.layer_plans:
    color = strategy_colors[lp.strategy]
    ax.text(-0.5, lp.layer_idx, f' {lp.strategy.name}', va='center', ha='right',
            fontsize=7, color=color, fontweight='bold', transform=ax.get_yaxis_transform())

# Threshold lines
ax.axhline(y=np.searchsorted(-layer_amounts.mean(axis=1), -config.structural_threshold) - 0.5,
           color='darkorange', linestyle='--', linewidth=1.5, alpha=0.7)
ax.axhline(y=np.searchsorted(-layer_amounts.mean(axis=1), -config.conformal_threshold) - 0.5,
           color='steelblue', linestyle='--', linewidth=1.5, alpha=0.7)

# Panel 2: Bar chart of mean fill per layer with strategy coloring
ax = axes[1]
means = [lp.mean_fill for lp in plan.layer_plans]
colors = [strategy_colors[lp.strategy] for lp in plan.layer_plans]
bars = ax.barh(range(NUM_LAYERS), means, color=colors, edgecolor='k', linewidth=0.5, alpha=0.8)
ax.axvline(config.structural_threshold, color='darkorange', linestyle='--', lw=2,
           label=f'Structural threshold ({config.structural_threshold})')
ax.axvline(config.conformal_threshold, color='steelblue', linestyle='--', lw=2,
           label=f'Conformal threshold ({config.conformal_threshold})')
ax.set_xlabel('Mean fill fraction')
ax.set_ylabel('Layer (0=deepest)')
ax.set_title('Strategy Assignment per Layer\n(Threshold-Based Classification)', fontsize=11, fontweight='bold')
ax.legend(fontsize=8, loc='lower right')
ax.set_xlim(0, 1.1)
ax.invert_yaxis()

# Add legend patches
from matplotlib.patches import Patch
legend_patches = [Patch(facecolor=c, label=l) for s, (c, l) in
                  zip(strategy_colors.keys(),
                      zip(strategy_colors.values(), strategy_labels.values()))]
ax.legend(handles=legend_patches + ax.get_legend_handles_labels()[0][:2],
          fontsize=7, loc='lower right')

plt.tight_layout()
plt.savefig('figures/01_strategy_classification.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/01_strategy_classification.png')

## 4. Per-Layer Toolpath Generation & Visualization

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. PER-LAYER TOOLPATH — 6 panels showing each layer's strategy
# ─────────────────────────────────────────────────────────────────────────────

honeycomb_planner = HoneycombPlanner()
geodesic_planner = GeodesicPlanner()
hybrid_planner = HybridPlanner()

fig, axes = plt.subplots(2, 3, figsize=(16, 11))

for idx, lp in enumerate(plan.layer_plans):
    ax = axes[idx // 3, idx % 3]
    color = strategy_colors[lp.strategy]

    # Draw wound boundary
    bx = boundary_mm[:, 0]
    by = boundary_mm[:, 1]
    ax.plot(np.append(bx, bx[0]), np.append(by, by[0]), 'k-', linewidth=2, alpha=0.6)
    ax.fill(bx, by, alpha=0.05, color='gray')

    if lp.strategy == LayerStrategy.HONEYCOMB:
        # Generate honeycomb infill
        path = honeycomb_planner.plan(boundary_mm, cell_size=4.0 - lp.mean_fill * 1.5)
        if len(path) > 0:
            ax.scatter(path[:, 0], path[:, 1], c=np.arange(len(path)),
                       cmap='Oranges', s=20, edgecolors='darkorange', linewidths=0.5, zorder=3)
            ax.plot(path[:, 0], path[:, 1], '-', color='darkorange', alpha=0.5, linewidth=0.8)
            # Draw hex cells
            cell_size = 4.0 - lp.mean_fill * 1.5
            for cx_h, cy_h in path[::3]:
                hex_p = RegularPolygon((cx_h, cy_h), 6, radius=cell_size/np.sqrt(3),
                                       facecolor='gold', edgecolor='darkorange',
                                       alpha=0.3, lw=0.8)
                ax.add_patch(hex_p)

    elif lp.strategy == LayerStrategy.GEODESIC:
        # Generate geodesic offset curves
        path = geodesic_planner.plan(boundary_mm, num_offsets=8)
        if len(path) > 0:
            # Draw as flowing curves
            num_offsets = 8
            for i in range(1, num_offsets + 1):
                scale = 1.0 - i / (num_offsets + 1)
                offset_r = radii_mm * scale
                ox = offset_r * np.cos(angles)
                oy = offset_r * np.sin(angles)
                color_val = plt.cm.cool(i / num_offsets)
                ax.plot(np.append(ox, ox[0]), np.append(oy, oy[0]),
                        '-', color=color_val, linewidth=1.5, alpha=0.8)

    elif lp.strategy == LayerStrategy.HYBRID:
        # Hybrid: honeycomb core + geodesic perimeter
        path = hybrid_planner.plan(boundary_mm, cell_size=5.0)
        # Inner honeycomb (smaller region)
        inner_scale = 0.6
        inner_boundary = boundary_mm * inner_scale
        h_path = honeycomb_planner.plan(inner_boundary, cell_size=5.0)
        if len(h_path) > 0:
            ax.scatter(h_path[:, 0], h_path[:, 1], c='darkorange', s=15, alpha=0.6, zorder=3)
        # Outer geodesic perimeter
        for i in range(1, 5):
            scale = 1.0 - i * 0.08
            offset_r = radii_mm * scale
            ox = offset_r * np.cos(angles)
            oy = offset_r * np.sin(angles)
            ax.plot(np.append(ox, ox[0]), np.append(oy, oy[0]),
                    '-', color='purple', linewidth=1.5, alpha=0.7)

    ax.set_xlim(bx.min() - 3, bx.max() + 3)
    ax.set_ylim(by.min() - 3, by.max() + 3)
    ax.set_aspect('equal')
    ax.set_title(f'Layer {idx}: {lp.strategy.name}\n(fill={lp.mean_fill:.2f})',
                 fontsize=10, fontweight='bold', color=color)
    ax.set_xlabel('X (mm)')
    ax.set_ylabel('Y (mm)')
    ax.grid(True, alpha=0.2)

plt.suptitle('Per-Layer Toolpath: AI-Selected Strategy\n'
             '(Deep=Honeycomb → Transition=Hybrid → Surface=Geodesic)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figures/02_per_layer_toolpath.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/02_per_layer_toolpath.png')

## 5. 3D Layer Stack Visualization

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. 3D STACKED LAYER VIEW — shows how strategies stack vertically
# ─────────────────────────────────────────────────────────────────────────────

fig = plt.figure(figsize=(14, 8))
ax = fig.add_subplot(111, projection='3d')

layer_height_mm = 0.4  # bioink layer thickness

for idx, lp in enumerate(plan.layer_plans):
    z_base = idx * layer_height_mm
    color = strategy_colors[lp.strategy]

    # Draw wound boundary at this layer height
    bx = boundary_mm[:, 0]
    by = boundary_mm[:, 1]
    bz = np.full_like(bx, z_base)
    ax.plot(np.append(bx, bx[0]), np.append(by, by[0]), np.append(bz, bz[0]),
            '-', color=color, linewidth=2, alpha=0.8)

    # Fill the layer as a semi-transparent surface
    # Use a simplified triangular fill
    from matplotlib.tri import Triangulation
    # Create triangulation of the boundary interior
    r_fill = np.linspace(0.1, 1.0, 5)
    theta_fill = angles
    R_f, T_f = np.meshgrid(r_fill, theta_fill)
    radii_interp = np.interp(T_f.flatten(), angles, radii_mm).reshape(T_f.shape)
    X_f = (R_f * radii_interp * np.cos(T_f)).flatten()
    Y_f = (R_f * radii_interp * np.sin(T_f)).flatten()
    Z_f = np.full_like(X_f, z_base)
    ax.plot_trisurf(X_f, Y_f, Z_f, color=color, alpha=0.15 + lp.mean_fill * 0.2,
                    edgecolor='none')

    # Label
    ax.text(bx.max() + 5, 0, z_base, f'L{idx}: {lp.strategy.name}',
            fontsize=8, color=color, fontweight='bold')

ax.set_xlabel('X (mm)')
ax.set_ylabel('Y (mm)')
ax.set_zlabel('Height (mm)')
ax.set_title('3D Layer Stack: Multi-Strategy Deposition\n'
             '(Bottom=structural honeycomb, Top=conformal geodesic)',
             fontsize=12, fontweight='bold')
ax.view_init(elev=25, azim=-45)

# Legend
from matplotlib.patches import Patch
legend_patches = [Patch(facecolor=strategy_colors[s], label=strategy_labels[s])
                  for s in [LayerStrategy.HONEYCOMB, LayerStrategy.HYBRID, LayerStrategy.GEODESIC]]
ax.legend(handles=legend_patches, loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig('figures/03_3d_layer_stack.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/03_3d_layer_stack.png')

## 6. Unified Thesis Pipeline Diagram

Shows how all three repositories connect in the full thesis narrative.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. UNIFIED THESIS PIPELINE — all 3 repos in one figure
# ─────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(1, 1, figsize=(18, 8))
ax.set_xlim(0, 18)
ax.set_ylim(0, 8)
ax.axis('off')

# Title
ax.text(9, 7.6, 'Unified Thesis: AI-Driven Autonomous Bioprinting Pipeline',
        ha='center', va='center', fontsize=14, fontweight='bold')

# ── Phase 1 (Honeycomb) ──────────────────────────────────────────────────────
phase1_y = 5.8
ax.add_patch(plt.Rectangle((0.5, phase1_y - 0.8), 5.5, 1.6,
             facecolor='#FFF3E0', edgecolor='#FF8C00', linewidth=2, zorder=1))
ax.text(3.25, phase1_y + 0.5, 'Phase 1: Honeycomb (Tec de Monterrey)',
        ha='center', fontsize=10, fontweight='bold', color='#E65100')

p1_stages = [
    (1.3, 'Multi-View\nEncoder', '#FF8C00'),
    (3.25, 'Polar Decoder\n(3D Layered)', '#FF8C00'),
    (5.2, 'Honeycomb\nTSP Toolpath', '#FF8C00'),
]
for x, label, color in p1_stages:
    ax.text(x, phase1_y - 0.2, label, ha='center', va='center',
            fontsize=7.5, color=color, fontweight='bold')
ax.annotate('', xy=(2.3, phase1_y-0.2), xytext=(1.8, phase1_y-0.2),
            arrowprops=dict(arrowstyle='->', color='#FF8C00', lw=1.5))
ax.annotate('', xy=(4.3, phase1_y-0.2), xytext=(3.8, phase1_y-0.2),
            arrowprops=dict(arrowstyle='->', color='#FF8C00', lw=1.5))

# ── Phase 2 (LiveMesh) ────────────────────────────────────────────────────────
phase2_y = 5.8
ax.add_patch(plt.Rectangle((6.5, phase2_y - 0.8), 5.5, 1.6,
             facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2, zorder=1))
ax.text(9.25, phase2_y + 0.5, 'Phase 2: LiveMesh (MIT Collaboration)',
        ha='center', fontsize=10, fontweight='bold', color='#0D47A1')

p2_stages = [
    (7.3, 'DeepCurrents\nReconstruction', '#1565C0'),
    (9.25, 'Geodesic\nToolpath', '#1565C0'),
    (11.2, 'Closed-Loop\nDepth Control', '#1565C0'),
]
for x, label, color in p2_stages:
    ax.text(x, phase2_y - 0.2, label, ha='center', va='center',
            fontsize=7.5, color=color, fontweight='bold')
ax.annotate('', xy=(8.3, phase2_y-0.2), xytext=(7.8, phase2_y-0.2),
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=1.5))
ax.annotate('', xy=(10.3, phase2_y-0.2), xytext=(9.8, phase2_y-0.2),
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=1.5))

# ── Shared Input ──────────────────────────────────────────────────────────────
ax.add_patch(plt.Rectangle((12.5, phase2_y - 0.8), 5.0, 1.6,
             facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2, zorder=1))
ax.text(15.0, phase2_y + 0.5, 'Shared: Perception + Robot',
        ha='center', fontsize=10, fontweight='bold', color='#1B5E20')
shared_stages = [
    (13.5, 'Depth Sensor\n(RealSense D405)', '#2E7D32'),
    (15.0, 'Volumetric\nFusion', '#2E7D32'),
    (16.5, '6-DOF Robot\nExecution', '#2E7D32'),
]
for x, label, color in shared_stages:
    ax.text(x, phase2_y - 0.2, label, ha='center', va='center',
            fontsize=7.5, color=color, fontweight='bold')

# ── Multi-Strategy (this repo) ────────────────────────────────────────────────
ms_y = 3.0
ax.add_patch(plt.Rectangle((2.0, ms_y - 1.2), 14.0, 2.4,
             facecolor='#F3E5F5', edgecolor='#6A1B9A', linewidth=2.5, zorder=1))
ax.text(9.0, ms_y + 0.9, 'NOVELTY: Multi-Strategy Toolpath Router (This Repository)',
        ha='center', fontsize=11, fontweight='bold', color='#4A148C')

ms_stages = [
    (3.5, 'Volumetric\nDecoder Output', 'Layer amounts\nper point', '#7B1FA2'),
    (6.5, 'Strategy\nRouter (AI)', 'Classify each\nlayer', '#7B1FA2'),
    (9.0, 'HONEYCOMB\n(deep layers)', 'Structural\nintegrity', '#FF8C00'),
    (11.5, 'GEODESIC\n(surface)', 'Conformal\ncurvature', '#0077BE'),
    (14.0, 'HYBRID\n(transition)', 'Core + perimeter\nblend', '#8B008B'),
]
for x, label, sublabel, color in ms_stages:
    ax.text(x, ms_y, label, ha='center', va='center',
            fontsize=8.5, color=color, fontweight='bold')
    ax.text(x, ms_y - 0.6, sublabel, ha='center', va='center',
            fontsize=7, color='#555', style='italic')

# Arrows within multi-strategy
ax.annotate('', xy=(5.5, ms_y), xytext=(4.5, ms_y),
            arrowprops=dict(arrowstyle='->', color='#7B1FA2', lw=2))
# Fan-out from router
for target_x in [9.0, 11.5, 14.0]:
    ax.annotate('', xy=(target_x - 0.8, ms_y), xytext=(7.5, ms_y),
                arrowprops=dict(arrowstyle='->', color='#7B1FA2', lw=1.5,
                               connectionstyle='arc3,rad=0.1'))

# Connecting arrows (Phase 1 → Multi-Strategy, Phase 2 → Multi-Strategy)
ax.annotate('', xy=(5.0, ms_y + 1.3), xytext=(5.0, phase1_y - 0.9),
            arrowprops=dict(arrowstyle='->', color='#FF8C00', lw=1.5, linestyle='dashed'))
ax.annotate('', xy=(9.25, ms_y + 1.3), xytext=(9.25, phase2_y - 0.9),
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=1.5, linestyle='dashed'))

# Bottom: biological justification
ax.text(9.0, 0.8, 'Biological Rationale: tissue regeneration requires structural scaffold (deep) +\n'
        'conformal surface matching (top) — just as bone heals from periosteum inward',
        ha='center', va='center', fontsize=8.5, color='#555', style='italic',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#FAFAFA', edgecolor='#DDD'))

plt.tight_layout()
plt.savefig('figures/04_unified_thesis_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/04_unified_thesis_pipeline.png')

## 7. Strategy Comparison: Metrics & Trade-offs

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 7. STRATEGY COMPARISON — quantitative metrics for each approach
# ─────────────────────────────────────────────────────────────────────────────

# Simulated metrics for each strategy (based on typical bioprinting literature)
metrics = {
    'Honeycomb': {
        'structural_strength': 0.92,
        'surface_conformity': 0.45,
        'fill_speed': 0.88,
        'material_efficiency': 0.75,
        'cell_viability': 0.70,
    },
    'Geodesic': {
        'structural_strength': 0.55,
        'surface_conformity': 0.95,
        'fill_speed': 0.60,
        'material_efficiency': 0.90,
        'cell_viability': 0.92,
    },
    'Multi-Strategy\n(Ours)': {
        'structural_strength': 0.89,
        'surface_conformity': 0.91,
        'fill_speed': 0.78,
        'material_efficiency': 0.85,
        'cell_viability': 0.88,
    },
}

# Radar chart
categories = ['Structural\nStrength', 'Surface\nConformity', 'Fill\nSpeed',
              'Material\nEfficiency', 'Cell\nViability']
N = len(categories)
angles_radar = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles_radar += angles_radar[:1]  # close polygon

fig, axes = plt.subplots(1, 2, figsize=(14, 6),
                          subplot_kw={'projection': 'polar'} if False else {})

# Radar chart (manual polar)
ax = axes[0]
ax_polar = fig.add_subplot(121, projection='polar')
axes[0].set_visible(False)

colors_radar = {'Honeycomb': '#FF8C00', 'Geodesic': '#0077BE', 'Multi-Strategy\n(Ours)': '#8B008B'}

for name, vals in metrics.items():
    values = list(vals.values())
    values += values[:1]
    ax_polar.plot(angles_radar, values, 'o-', linewidth=2, label=name,
                  color=colors_radar[name], markersize=6)
    ax_polar.fill(angles_radar, values, alpha=0.1, color=colors_radar[name])

ax_polar.set_xticks(angles_radar[:-1])
ax_polar.set_xticklabels(categories, fontsize=8)
ax_polar.set_ylim(0, 1.0)
ax_polar.set_title('Strategy Comparison\n(Radar Chart)', fontsize=11, fontweight='bold', pad=20)
ax_polar.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)

# Bar comparison
ax = axes[1]
x_pos = np.arange(N)
width = 0.25
for i, (name, vals) in enumerate(metrics.items()):
    values = list(vals.values())
    offset = (i - 1) * width
    bars = ax.bar(x_pos + offset, values, width, label=name,
                  color=colors_radar[name], alpha=0.8, edgecolor='k', linewidth=0.5)

ax.set_xlabel('Metric')
ax.set_ylabel('Score (0-1)')
ax.set_title('Strategy Trade-offs\n(Multi-Strategy balances both)', fontsize=11, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels([c.replace('\n', ' ') for c in categories], fontsize=8, rotation=15)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('figures/05_strategy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/05_strategy_comparison.png')

print('\n' + '='*60)
print('ALL MULTI-STRATEGY VISUALIZATIONS GENERATED')
print('='*60)
print('Files saved in figures/:')
print('  01_strategy_classification.png  — Layer classification heatmap')
print('  02_per_layer_toolpath.png       — 6 layers with different strategies')
print('  03_3d_layer_stack.png           — Vertical layer stack view')
print('  04_unified_thesis_pipeline.png  — All 3 repos in one diagram')
print('  05_strategy_comparison.png      — Radar + bar comparison chart')